# Defining and profiling a stabilizer code

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we specify a stabilizer code in qodec and query its algebraic error-detection structure?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Define a stabilizer code and its logical Pauli basis with `qodec.Code`.
- Query code parameters, syndromes, and logical operator effects with `ec.CodeProfile`.
- Evaluate code distance under single-qubit, Pauli-biased, and correlated multi-qubit error models.

## Encoding logical qubits in the C4 code

Our primary running example is the four-qubit $[[4, 2, 2]]$ **`C4` code** on physical qubits `0, 1, 2, 3`, defined by stabilizers $S_0 = X_0 X_1 X_2 X_3$ and $S_1 = Z_0 Z_1 Z_2 Z_3$ with logical Pauli pairs $\bar{X}_0 = X_0 X_1,\; \bar{Z}_0 = Z_0 Z_2$ and $\bar{X}_1 = X_0 X_2,\; \bar{Z}_1 = Z_0 Z_1$.

`qodec.Code` specifies Pauli generators using indexed factors `P_i` ($P \in \{X, Y, Z\}$ on physical qubit `i`, omitting identity factors). `qodec.Code` itself is a declarative specification; wrapping it in `ec.CodeProfile` takes an immutable snapshot, validates stabilizer/logical commutation relations, and exposes algebraic query and distance solvers:

In [1]:
import qodec
from qdk import ec

c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

c4_profile = ec.CodeProfile(c4)
print("Physical support :", sorted(c4_profile.support))
print("Physical qubits  :", c4_profile.length)
print("Logical qubits   :", c4_profile.logical_qubit_count)
print("Stabilizers      :", list(c4_profile.stabilizers))
print("Logical X        :", list(c4_profile.x))
print("Logical Z        :", list(c4_profile.z))

Physical support : [0, 1, 2, 3]
Physical qubits  : 4
Logical qubits   : 2
Stabilizers      : [XXXX, ZZZZ]
Logical X        : [XX, XIX]
Logical Z        : [ZIZ, ZZ]


## Computing code distance and inspecting syndromes

Calling `c4_profile.distance()` searches for a minimum-weight error that commutes with every stabilizer while acting nontrivially on the logical subspace. By default, every single-qubit $X$, $Y$, or $Z$ error counts as one fault event.

For any Pauli operator `E`, `CodeProfile` exposes three algebraic queries (formatting dense logical Paulis with trailing identities omitted, so `X` on two logical qubits denotes $X \otimes I$ vs. `IX` for $I \otimes X$):
- `syndrome_of(E)` returns the 0-based indices of anticommuting stabilizers in `code.stabilizers`.
- `logical_effect_of(E)` returns the signed logical Pauli induced on the code space when `E` commutes with all stabilizers (raising `ValueError` otherwise). Passing `including_phase=False` instead computes the unsigned frame projection from anticommutation with the declared `z`/`x` generators even when `syndrome_of(E)` is nonempty.
- `is_logical(E)` returns `True` when `syndrome_of(E)` is empty and `logical_effect_of(E)` is non-identity.

In [2]:
distance = c4_profile.distance()
witness = distance.witness.product
print(f"Code distance: {distance}  (witness: {witness})")
assert distance.value == 2 and c4_profile.is_logical(witness)

for label in ["X_1", "Z_2", "X_0 X_1", "X_2 X_3", "X_0 X_1 X_2 X_3"]:
    err = ec.Pauli(label)
    syn = sorted(c4_profile.syndrome_of(err))
    exact = str(c4_profile.logical_effect_of(err)) if not syn else "ValueError"
    proj = str(c4_profile.logical_effect_of(err, including_phase=False))
    print(f"{label:<16} -> syn={str(syn):<6} exact={exact:<11} frame_proj={proj}")

Code distance: 2  (witness: XX)
X_1              -> syn=[1]    exact=ValueError  frame_proj=IX
Z_2              -> syn=[0]    exact=ValueError  frame_proj=IZ
X_0 X_1          -> syn=[]     exact=X           frame_proj=X
X_2 X_3          -> syn=[]     exact=X           frame_proj=X
X_0 X_1 X_2 X_3  -> syn=[]     exact=I           frame_proj=I


Notice the contrast on `X_1`: because `X_1` anticommutes with $Z_0 Z_1 Z_2 Z_3$ (`syn=[1]`), it maps states out of the code space and has no signed code-space action (`exact=ValueError`). Passing `including_phase=False` still reports `IX` because `X_1` anticommutes with the chosen representative `z[1] = Z_0 Z_1`—useful when tracking Pauli frame coordinates, though `is_logical(X_1)` remains `False`.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does calling logical_effect_of on X_1 raise ValueError by default, and what does including_phase=False return instead?</summary>

<div style="padding:0.1em 1em;">

By default (`including_phase=True`), `logical_effect_of` requires `E` to commute with every stabilizer so its signed action on the code space is well-defined; because `X_1` anticommutes with `Z_0 Z_1 Z_2 Z_3`, it raises `ValueError`. With `including_phase=False`, `CodeProfile` returns the unsigned frame projection `IX` (from anticommutation with `z[1] = Z_0 Z_1`), which depends on the chosen logical representatives and does not imply an undetected logical error.

</div>

</details></div>

## Biased and correlated error models

Physical platforms frequently exhibit Pauli bias or multi-qubit crosstalk bursts rather than i.i.d. depolarizing noise. `CodeProfile.distance(errors=...)` accepts either a Pauli-letter filter (`"X"`, `"Z"`, `"XZ"`) or an explicit sequence of `ec.Pauli` fault generators—minimizing the **number of supplied generators** whose product forms an undetected nontrivial logical operator (while `distance_bounds(upper_bound=...)` caps search depth on larger codes).

Below, we define the non-CSS $[[5, 1, 3]]$ cyclic five-qubit code (`C5`) and compare its general single-qubit distance against pure-$X$ noise:

In [3]:
from _unit import make_five_qubit_code

five_qubit_code = qodec.Code(
    "C5",
    stabilizers=[
        "X_0 Z_1 Z_2 X_3",
        "X_1 Z_2 Z_3 X_4",
        "X_0 X_2 Z_3 Z_4",
        "Z_0 X_1 X_3 Z_4",
    ],
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["Z_0 Z_1 Z_2 Z_3 Z_4"],
)
assert list(five_qubit_code.stabilizers) == list(make_five_qubit_code().stabilizers)

five_profile = ec.CodeProfile(five_qubit_code)
d_all = five_profile.distance()
d_x = five_profile.distance(errors="X")
print(f"[[5, 1, 3]] general distance: {d_all}  (witness: {d_all.witness.product})")
print(f"[[5, 1, 3]] pure-X distance : {d_x}  (witness: {d_x.witness.product})")

[[5, 1, 3]] general distance: 3  (witness: IIZXZ)
[[5, 1, 3]] pure-X distance : 5  (witness: XXXXX)


<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does restricting errors to pure X raise the five-qubit code's distance from 3 to 5?</summary>

<div style="padding:0.1em 1em;">

Every weight-three logical operator of the $[[5, 1, 3]]$ code mixes $X$ and $Z$ factors (such as `Z_0 X_2 Z_3`). When only $X$ errors can occur, those mixed-Pauli operators cannot form, leaving weight-five `X_0 X_1 X_2 X_3 X_4` as the minimum-weight logical error.

</div>

</details></div>

## Exercise: Effective pure-X distance under nearest-neighbor crosstalk

Suppose a five-qubit ring suffers single-qubit bit flips $X_i$ **and** correlated nearest-neighbor crosstalk bursts $X_i X_{(i+1) \bmod 5}$ ($i \in \{0, 1, 2, 3, 4\}$), where each single-qubit flip or adjacent pair counts as one fault event. Complete `five_qubit_correlated_x_distance` to pass this explicit `list[ec.Pauli]` to `CodeProfile(code).distance(errors=...)` and return the resulting integer fault-event distance.

In [ ]:
from qdk import ec
from _unit import exercise, make_five_qubit_code


@exercise
def five_qubit_correlated_x_distance() -> int:
    code = make_five_qubit_code()
    single_x = [ec.Pauli(f'X_{i}') for i in range(5)]
    # Add the 5 nearest-neighbor bursts X_i X_{(i+1)%5} and return distance(errors=...).value
    return ...

**Hint**

Build `adj_xx = [ec.Pauli(f'X_{i} X_{(i + 1) % 5}') for i in range(5)]` and return `ec.CodeProfile(code).distance(errors=single_x + adj_xx).value`.

In [ ]:
from qdk import ec
from _unit import exercise, make_five_qubit_code


@exercise
def five_qubit_correlated_x_distance() -> int:
    code = make_five_qubit_code()
    single_x = [ec.Pauli(f'X_{i}') for i in range(5)]
    adj_xx = [ec.Pauli(f'X_{i} X_{(i + 1) % 5}') for i in range(5)]
    return ec.CodeProfile(code).distance(errors=single_x + adj_xx).value

`3`. Under pure-$X$ errors, the only nontrivial logical operator on `C5` is $X_0 X_1 X_2 X_3 X_4$. With adjacent two-qubit bursts available as single fault events, three events—such as $(X_0 X_1)(X_2 X_3)(X_4)$—realize that weight-five logical operator, reducing the pure-$X$ fault distance from `5` to `3`.

## Further reading

- [Develop and validate a QEC protocol ↗](https://learn.microsoft.com/en-us/azure/quantum/develop-validate-qec-protocol)
- [Code definitions in qodec ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/concepts/code.md)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../00-overview/overview.workbook.ipynb" title="Previous unit: Tutorial Overview" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Tutorial Overview</span></span></a><a href="../02-instruction-sets/instruction_sets.workbook.ipynb" title="Next unit: Building a Baseline Qodec and Instruction Sets" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Building a Baseline Qodec and Instruction Sets</span></span><span aria-hidden="true">&#8250;</span></a></div>